# NB08 — Reextracción con pooling adaptativo (2×2)

## Propósito

Reextraer las representaciones de AsymMirai sobre los 5.000 estudios de VinDr-Mammo, pero esta vez **preservando información espacial** mediante `AdaptiveAvgPool2d((2,2)) + AdaptiveMaxPool2d((2,2))` en lugar del GAP+GMP global del NB03.

## Motivación

El backbone de AsymMirai devuelve un mapa `(512, 52, 64)` por vista. En el NB03 lo reducimos a 1024 dims (512 GAP + 512 GMP), descartando toda la información espacial. Pero los hallazgos BI-RADS sospechosos (calcificaciones, masas, asimetrías focales) son **focales** por naturaleza — ocupan unos pocos píxeles latentes de los 52×64=3328 disponibles. Promediar sobre todo el campo de visión los diluye; el máximo global los captura como pico pero pierde la información de localización.

El pooling adaptativo a una rejilla `(2,2)` divide el mapa latente en **cuatro cuadrantes** (superior-izquierdo, superior-derecho, inferior-izquierdo, inferior-derecho) y calcula el promedio y el máximo dentro de cada cuadrante. Conserva localización gruesa (en qué cuadrante está el pico) sin disparar la dimensionalidad como lo haría una rejilla más fina.

## Decisiones de diseño

- **Rejilla (2,2)** en lugar de (4,4) o (7,7): compromiso entre conservar información espacial y no inflar la dimensionalidad. Con (4,4) tendríamos 16384 dims por vista vs 4096 con (2,2). Dado que el training a nivel mama tiene ~3200 muestras (~395 positivos), (4,4) es probable que sobreajuste.
- **Avg + Max** en lugar de solo Avg o solo Max: igual razonamiento que en NB03 — Avg captura magnitud distribuida, Max captura picos. Conservar ambos no encarece sensiblemente.
- **Mismo preprocesado DICOM** que en NB03 (MIRAI_MEAN/STD, alineación por centroide, redimensionado a 1664×2048): la única diferencia entre NB03 y NB08 es la función de pooling.
- **Mismo metadata.csv**: las etiquetas y splits no cambian. Solo regeneramos las features.

## Outputs

| Archivo | Shape | Tamaño |
|---|---|---|
| `X_view_22.npy` | `(4999, 4, 4096)` | ~328 MB |
| `X_asym_22.npy` | `(4999, 2, 4096)` | ~164 MB |
| `done_studies_22.txt` | 4999 líneas | <1 MB |

El `metadata.csv` original del NB03 se reutiliza tal cual; no se sobrescribe.

## Tiempo estimado

~45-50 minutos en GPU RTX 4070 Super (similar al NB03). El cuello de botella es la lectura DICOM y el forward del backbone, no el pooling.

In [1]:
import os, sys, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
import pydicom
from pathlib import Path

# Raíz del proyecto: por defecto, la carpeta padre de notebooks/.
# Sobrescribible con la variable de entorno TFM_PROJECT_ROOT.
BASE      = os.environ.get('TFM_PROJECT_ROOT',
                           os.path.abspath(os.path.join(os.getcwd(), '..')))
ASYMMIRAI    = os.path.join(BASE, 'AsymMirai')
DATA         = os.path.join(BASE, 'Data', 'vindr-mammo')
OUTPUTS      = os.path.join(BASE, 'outputs')
WEIGHTS      = os.path.join(ASYMMIRAI, 'snapshots', 'trained_asymmirai.pt')
IMG_DIR      = os.path.join(DATA, 'images')
BREAST_CSV   = os.path.join(DATA, 'breast-level_annotations.csv')
FEATURES_DIR = os.path.join(OUTPUTS, 'Features')
Path(FEATURES_DIR).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, ASYMMIRAI)
sys.path.insert(0, os.path.join(ASYMMIRAI, 'asymmetry_model'))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Salida: {FEATURES_DIR}')

Device: cuda
Salida: C:\Users\victo\Documents\TFM\Proyecto\Outputs\Features


## 1. Carga del modelo congelado y stretch params

Idéntico al NB03. El modelo y los pesos no cambian.

In [2]:
model = torch.load(WEIGHTS, map_location=DEVICE, weights_only=False)
model.eval()
for p in model.parameters():
    p.requires_grad = False

USE_STRETCH = getattr(model, 'use_stretch', False)
cc_stretch  = model.cc_stretch_params.detach() if USE_STRETCH else None
mlo_stretch = model.mlo_stretch_params.detach() if USE_STRETCH else None
print(f'use_stretch = {USE_STRETCH}    alignment_space = {getattr(model,"alignment_space",None)}')

c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\torch\serialization.py:1782: SourceChangeWarning: source code of class 'mirai_localized_dif_head.LocalizedDifModel' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\torch\serialization.py:1782: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)


use_stretch = True    alignment_space = None


## 2. Funciones de preprocesado y extracción

**Lo único que cambia respecto al NB03 es `pool_features`.** Todo lo demás (load_dicom, align_by_centroid, preprocess_view, compute_asym, extract_study) es idéntico para garantizar que cualquier diferencia entre las features de NB03 y NB08 venga **únicamente del pooling**.

### Nuevo pooling

```python
# NB03 (GAP+GMP global)
gap = x.mean(dim=(-2, -1))     # (B, C)        → 512 dims
gmp = x.amax(dim=(-2, -1))     # (B, C)        → 512 dims
concat → 1024 dims por vista

# NB08 (AdaptiveAvg + AdaptiveMax a 2×2)
avg22 = F.adaptive_avg_pool2d(x, (2,2))   # (B, C, 2, 2)
max22 = F.adaptive_max_pool2d(x, (2,2))   # (B, C, 2, 2)
flatten → 2048 + 2048 = 4096 dims por vista
```

In [3]:
MIRAI_MEAN = 7699.5
MIRAI_STD  = 11765.06
TARGET_H, TARGET_W = 1664, 2048

def load_dicom(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    pixels = pixels * float(getattr(ds, 'RescaleSlope', 1)) + float(getattr(ds, 'RescaleIntercept', 0))
    if hasattr(ds, 'WindowCenter') and hasattr(ds, 'WindowWidth'):
        wc = float(ds.WindowCenter[0]) if hasattr(ds.WindowCenter, '__iter__') else float(ds.WindowCenter)
        ww = float(ds.WindowWidth[0])  if hasattr(ds.WindowWidth,  '__iter__') else float(ds.WindowWidth)
        pixels = np.clip(pixels, wc-ww/2, wc+ww/2)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        pixels = pixels.max() - pixels
    return pixels

def align_by_centroid(img):
    h, w = img.shape
    mask = (img > img.mean()).astype(np.uint8)
    M = cv2.moments(mask)
    if M['m00'] == 0:
        return img
    cy, cx = int(M['m01']/M['m00']), int(M['m10']/M['m00'])
    M_aff = np.float32([[1, 0, w//2-cx], [0, 1, h//2-cy]])
    return cv2.warpAffine(img, M_aff, (w, h), borderValue=0)

def preprocess_view(dcm_path):
    img = load_dicom(dcm_path)
    img = align_by_centroid(img)
    img = cv2.resize(img, (TARGET_W, TARGET_H), interpolation=cv2.INTER_LINEAR)
    img = (img - MIRAI_MEAN) / MIRAI_STD
    img = np.stack([img, img, img], axis=0)[None, ...]
    return torch.from_numpy(img).float()

def pool_features_22(x):
    """
    AdaptiveAvgPool2d((2,2)) + AdaptiveMaxPool2d((2,2)) sobre (B, C, H, W).
    Devuelve (B, 4*C) con [avg_q1, avg_q2, avg_q3, avg_q4, max_q1, max_q2, max_q3, max_q4] por canal,
    donde q1..q4 son los 4 cuadrantes del mapa latente.
    Para C=512 -> salida (B, 4096).
    """
    avg = F.adaptive_avg_pool2d(x, (2, 2))   # (B, C, 2, 2)
    mx  = F.adaptive_max_pool2d(x, (2, 2))   # (B, C, 2, 2)
    # Aplanamos manteniendo (canal, cuadrante) para que el orden sea consistente
    avg_flat = avg.reshape(avg.size(0), -1)  # (B, C*4)
    mx_flat  = mx.reshape(mx.size(0), -1)    # (B, C*4)
    return torch.cat([avg_flat, mx_flat], dim=1)  # (B, C*8)

def compute_asym(L, R, stretch_p=None):
    """Mismo cálculo del Punto B que en NB02/NB03: |stretch ⊙ L − stretch ⊙ R|."""
    if stretch_p is not None:
        sp = stretch_p.view(1, -1, 1, 1).to(L.device)
        L = sp * L
        R = sp * R
    return torch.abs(L - R)

def extract_study(vistas):
    """
    vistas: dict {('L','CC'): path, ('R','CC'): path, ('L','MLO'): path, ('R','MLO'): path}
    Devuelve:
      X_view: (4, 4096) en orden [L-CC, R-CC, L-MLO, R-MLO]
      X_asym: (2, 4096) en orden [asym-CC, asym-MLO]
      o None si fallo en preprocesado.
    """
    try:
        l_cc  = preprocess_view(vistas[('L','CC')]).to(DEVICE)
        r_cc  = preprocess_view(vistas[('R','CC')]).to(DEVICE)
        l_mlo = preprocess_view(vistas[('L','MLO')]).to(DEVICE)
        r_mlo = preprocess_view(vistas[('R','MLO')]).to(DEVICE)
    except Exception as e:
        print(f'    [err preprocesado] {e}')
        return None

    with torch.no_grad():
        emb_l_cc  = model.backbone(l_cc)
        emb_r_cc  = model.backbone(r_cc)
        emb_l_mlo = model.backbone(l_mlo)
        emb_r_mlo = model.backbone(r_mlo)

        # Punto A: features de las 4 vistas con pooling (2,2)
        feat_l_cc  = pool_features_22(emb_l_cc)[0]    # (4096,)
        feat_r_cc  = pool_features_22(emb_r_cc)[0]
        feat_l_mlo = pool_features_22(emb_l_mlo)[0]
        feat_r_mlo = pool_features_22(emb_r_mlo)[0]

        # Punto B: asimetría bilateral con stretch, también con pooling (2,2)
        asym_cc  = compute_asym(emb_l_cc,  emb_r_cc,  cc_stretch  if USE_STRETCH else None)
        asym_mlo = compute_asym(emb_l_mlo, emb_r_mlo, mlo_stretch if USE_STRETCH else None)
        feat_asym_cc  = pool_features_22(asym_cc)[0]   # (4096,)
        feat_asym_mlo = pool_features_22(asym_mlo)[0]

    X_view = torch.stack([feat_l_cc, feat_r_cc, feat_l_mlo, feat_r_mlo], dim=0).cpu().numpy()  # (4, 4096)
    X_asym = torch.stack([feat_asym_cc, feat_asym_mlo], dim=0).cpu().numpy()                   # (2, 4096)
    return X_view, X_asym

## 3. Sanity check sobre un estudio antes de lanzar

Cogemos un estudio cualquiera y comprobamos que los shapes son los esperados antes de gastar 45 minutos en GPU.

In [4]:
df = pd.read_csv(BREAST_CSV)
primer_sid = df['study_id'].iloc[0]
df_test = df[df.study_id == primer_sid]
vistas_test = {(r.laterality, r.view_position): os.path.join(IMG_DIR, primer_sid, f'{r.image_id}.dicom')
               for _, r in df_test.iterrows()}

if all(os.path.isfile(p) for p in vistas_test.values()):
    t0 = time.time()
    result = extract_study(vistas_test)
    if result is not None:
        Xv, Xa = result
        print(f'OK ({time.time()-t0:.1f}s)')
        print(f'  X_view shape: {Xv.shape}   esperado (4, 4096)')
        print(f'  X_asym shape: {Xa.shape}   esperado (2, 4096)')
        print(f'  X_view rango: [{Xv.min():.4f}, {Xv.max():.4f}]   media: {Xv.mean():.4f}')
        print(f'  X_asym rango: [{Xa.min():.4f}, {Xa.max():.4f}]   media: {Xa.mean():.4f}')
        assert Xv.shape == (4, 4096), 'Shape de X_view incorrecto'
        assert Xa.shape == (2, 4096), 'Shape de X_asym incorrecto'
        print('\nSanity check OK. Listo para extracción masiva.')
    else:
        print('Fallo en extract_study sobre el primer estudio')
else:
    print('Algunas vistas del primer estudio no existen en disco; salta este sanity check')

OK (0.8s)
  X_view shape: (4, 4096)   esperado (4, 4096)
  X_asym shape: (2, 4096)   esperado (2, 4096)
  X_view rango: [0.0000, 2.7262]   media: 0.9349
  X_asym rango: [0.0000, 2.8481]   media: 0.0050

Sanity check OK. Listo para extracción masiva.


## 4. Construir las etiquetas (idéntico a NB03)

Reutilizamos exactamente la lógica de agregación a nivel mama y estudio del NB03. Si ya tienes `metadata.csv` del NB03, simplemente lo lees y saltas esta celda.

In [5]:
META_PATH = os.path.join(FEATURES_DIR, 'metadata.csv')

if os.path.isfile(META_PATH):
    study_agg = pd.read_csv(META_PATH)
    print(f'metadata.csv ya existe ({len(study_agg)} filas), reutilizando.')
    # Reconstruir el diccionario de paths a partir del CSV original de VinDr
    study_imgs = {}
    for sid, grp in df.groupby('study_id'):
        study_imgs[sid] = {(r.laterality, r.view_position): r.image_id for _, r in grp.iterrows()}
else:
    print('metadata.csv no existe, generándolo de cero...')
    
    def parse_birads(v):
        if isinstance(v, str):
            s = v.replace('BI-RADS', '').replace('BIRADS', '').strip()
            try: return int(s)
            except: return np.nan
        return int(v) if not pd.isna(v) else np.nan
    
    def parse_density(v):
        if isinstance(v, str):
            return v.replace('DENSITY', '').strip()
        return v
    
    df['birads_int'] = df['breast_birads'].apply(parse_birads)
    df['density']    = df['breast_density'].apply(parse_density)
    
    birads_breast = df.groupby(['study_id','laterality']).agg(
        birads  = ('birads_int', 'max'),
        density = ('density', lambda s: s.mode().iloc[0] if len(s.mode()) else 'C'),
    ).reset_index()
    birads_L = birads_breast[birads_breast.laterality == 'L'].set_index('study_id')
    birads_R = birads_breast[birads_breast.laterality == 'R'].set_index('study_id')
    
    study_agg = df.groupby('study_id').agg(
        max_birads = ('birads_int', 'max'),
        density    = ('density', lambda s: s.mode().iloc[0] if len(s.mode()) else 'C'),
        split      = ('split', 'first'),
    ).reset_index()
    study_agg['birads_L']  = study_agg['study_id'].map(birads_L['birads'])
    study_agg['birads_R']  = study_agg['study_id'].map(birads_R['birads'])
    study_agg['density_L'] = study_agg['study_id'].map(birads_L['density'])
    study_agg['density_R'] = study_agg['study_id'].map(birads_R['density'])
    study_agg['y_estudio'] = (study_agg['max_birads'] >= 4).astype(int)
    study_agg['y_L']       = (study_agg['birads_L'] >= 4).astype(int)
    study_agg['y_R']       = (study_agg['birads_R'] >= 4).astype(int)
    
    study_imgs = {}
    for sid, grp in df.groupby('study_id'):
        study_imgs[sid] = {(r.laterality, r.view_position): r.image_id for _, r in grp.iterrows()}
    
    print(f'Generadas {len(study_agg)} filas de metadata.')

print(f'\nEstudios: {len(study_agg)}')
print(f'Positivos nivel estudio: {study_agg.y_estudio.sum()}  ({100*study_agg.y_estudio.mean():.2f}%)')
n_mamas = len(study_agg)*2; n_pos_mamas = study_agg.y_L.sum() + study_agg.y_R.sum()
print(f'Positivos nivel mama: {n_pos_mamas} de {n_mamas} ({100*n_pos_mamas/n_mamas:.2f}%)')

metadata.csv ya existe (4999 filas), reutilizando.

Estudios: 4999
Positivos nivel estudio: 481  (9.62%)
Positivos nivel mama: 494 de 9998 (4.94%)


## 5. Lógica de reanudación

Igual que en NB03, leemos `done_studies_22.txt` si existe y saltamos lo ya procesado. Esto permite parar con Ctrl+C y continuar sin perder trabajo.

In [6]:
OUT_VIEW = os.path.join(FEATURES_DIR, 'X_view_22.npy')
OUT_ASYM = os.path.join(FEATURES_DIR, 'X_asym_22.npy')
OUT_DONE = os.path.join(FEATURES_DIR, 'done_studies_22.txt')

done = set()
if os.path.isfile(OUT_DONE):
    with open(OUT_DONE) as f:
        done = set(l.strip() for l in f if l.strip())
    print(f'Reanudando: ya hechos {len(done)} estudios')

# Usamos la misma lista de estudios que metadata.csv (en el mismo orden)
# para garantizar que la fila i de X_view_22 corresponde a la fila i de metadata.csv
pendientes = [s for s in study_agg.study_id.tolist() if s not in done]
print(f'Pendientes: {len(pendientes)}')

Pendientes: 4999


## 6. Bucle principal

Igual estructura que NB03: cada 100 estudios procesados, vuelca a disco para que un corte de luz no pierda nada. ETA mostrado cada 10 estudios.

In [7]:
X_view_buf, X_asym_buf = [], []
done_list = []
SAVE_EVERY = 100

# Precargar buffers si hay parciales
if len(done) > 0:
    if os.path.isfile(OUT_VIEW):
        X_view_buf = list(np.load(OUT_VIEW))
        X_asym_buf = list(np.load(OUT_ASYM))
    with open(OUT_DONE) as f:
        done_list = [l.strip() for l in f if l.strip()]
    print(f'Buffers precargados con {len(done_list)} estudios')

def flush():
    """Vuelca los buffers a disco."""
    np.save(OUT_VIEW, np.array(X_view_buf, dtype=np.float32))
    np.save(OUT_ASYM, np.array(X_asym_buf, dtype=np.float32))
    with open(OUT_DONE, 'w') as f:
        for sid in done_list:
            f.write(sid + '\n')

t_global = time.time()
errores = []

for i, sid in enumerate(pendientes, start=1):
    imgs = study_imgs.get(sid, {})
    required = [('L','CC'), ('R','CC'), ('L','MLO'), ('R','MLO')]
    if not all(k in imgs for k in required):
        errores.append((sid, 'faltan vistas en CSV')); continue
    vistas = {k: os.path.join(IMG_DIR, sid, f'{imgs[k]}.dicom') for k in required}
    if not all(os.path.isfile(p) for p in vistas.values()):
        errores.append((sid, 'DICOM ausente en disco')); continue

    t0 = time.time()
    result = extract_study(vistas)
    if result is None:
        errores.append((sid, 'fallo en extract')); continue
    X_view, X_asym = result

    X_view_buf.append(X_view)
    X_asym_buf.append(X_asym)
    done_list.append(sid)

    if i % 10 == 0 or i == 1:
        elapsed = time.time() - t_global
        rate = elapsed / i
        eta = rate * (len(pendientes) - i) / 60
        print(f'  [{i:5d}/{len(pendientes)}] sid={sid[:12]}...  {time.time()-t0:.1f}s/est  ETA {eta:.1f}min')

    if i % SAVE_EVERY == 0:
        flush()
        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

flush()
print(f'\nExtracción completa. Procesados {len(done_list)} estudios. Errores: {len(errores)}')
if errores:
    print('Primeros errores:')
    for sid, err in errores[:5]:
        print(f'  {sid[:12]}...  {err}')

  [    1/4999] sid=0025a5dc99fd...  0.5s/est  ETA 43.3min
  [   10/4999] sid=008b8e61390f...  0.5s/est  ETA 43.2min
  [   20/4999] sid=00fe39c6d128...  0.5s/est  ETA 44.7min
  [   30/4999] sid=017ff0405a2e...  0.5s/est  ETA 43.2min
  [   40/4999] sid=01e354f1ba4b...  0.4s/est  ETA 41.8min
  [   50/4999] sid=0249157fa6d1...  0.6s/est  ETA 41.8min
  [   60/4999] sid=02f802e21fa4...  0.5s/est  ETA 41.6min
  [   70/4999] sid=039e97685515...  0.5s/est  ETA 41.6min
  [   80/4999] sid=041656eea2b3...  0.6s/est  ETA 41.4min
  [   90/4999] sid=0495f34b7e81...  0.5s/est  ETA 41.4min
  [  100/4999] sid=04fb5102b72c...  0.6s/est  ETA 41.2min
  [  110/4999] sid=0575060d13f8...  0.6s/est  ETA 41.3min
  [  120/4999] sid=06394402b32d...  0.5s/est  ETA 41.3min
  [  130/4999] sid=0676f4386325...  0.5s/est  ETA 41.1min
  [  140/4999] sid=06fb1049bdeb...  0.3s/est  ETA 41.0min
  [  150/4999] sid=07889ee9b57d...  0.5s/est  ETA 41.0min
  [  160/4999] sid=07d6c93ac36d...  0.3s/est  ETA 40.7min
  [  170/4999]

## 7. Verificación final

Comprobamos shapes y consistencia con el metadata.csv (las filas deben estar en el mismo orden que en NB03).

In [8]:
X_view_22 = np.load(OUT_VIEW)
X_asym_22 = np.load(OUT_ASYM)

print(f'X_view_22: shape={X_view_22.shape}   {X_view_22.nbytes/1e6:.1f} MB')
print(f'X_asym_22: shape={X_asym_22.shape}   {X_asym_22.nbytes/1e6:.1f} MB')
print(f'\nMetadata filas: {len(study_agg)}')

assert X_view_22.shape[0] == len(study_agg), 'Desalineación entre X_view_22 y metadata'
assert X_asym_22.shape[0] == len(study_agg), 'Desalineación entre X_asym_22 y metadata'
assert X_view_22.shape[1:] == (4, 4096), f'Shape de X_view_22 inesperado: {X_view_22.shape}'
assert X_asym_22.shape[1:] == (2, 4096), f'Shape de X_asym_22 inesperado: {X_asym_22.shape}'

# Estadísticas comparativas: ¿se ve diferencia respecto a GAP+GMP del NB03?
X_view_orig = np.load(os.path.join(FEATURES_DIR, 'X_view.npy'))
X_asym_orig = np.load(os.path.join(FEATURES_DIR, 'X_asym.npy'))

print(f'\n--- Comparación de magnitudes (GAP+GMP vs pooling 2,2) ---')
print(f'X_view original  (GAP+GMP, 1024 dims): media={X_view_orig.mean():.4f}  std={X_view_orig.std():.4f}  max={X_view_orig.max():.4f}')
print(f'X_view nuevo     (pool2x2, 4096 dims): media={X_view_22.mean():.4f}  std={X_view_22.std():.4f}  max={X_view_22.max():.4f}')
print(f'X_asym original  (GAP+GMP, 1024 dims): media={X_asym_orig.mean():.4f}  std={X_asym_orig.std():.4f}  max={X_asym_orig.max():.4f}')
print(f'X_asym nuevo     (pool2x2, 4096 dims): media={X_asym_22.mean():.4f}  std={X_asym_22.std():.4f}  max={X_asym_22.max():.4f}')

print('\nTodo OK. Ya puedes proceder a NB09 (zoo de cabezas con pipeline unificado).')

X_view_22: shape=(4999, 4, 4096)   327.6 MB
X_asym_22: shape=(4999, 2, 4096)   163.8 MB

Metadata filas: 4999

--- Comparación de magnitudes (GAP+GMP vs pooling 2,2) ---
X_view original  (GAP+GMP, 1024 dims): media=0.9661  std=0.4652  max=8.8575
X_view nuevo     (pool2x2, 4096 dims): media=0.9395  std=0.4531  max=8.8575
X_asym original  (GAP+GMP, 1024 dims): media=0.0128  std=0.1245  max=10.1893
X_asym nuevo     (pool2x2, 4096 dims): media=0.0090  std=0.0940  max=10.1893

Todo OK. Ya puedes proceder a NB09 (zoo de cabezas con pipeline unificado).


## Siguiente paso

`NB09 — Pipeline unificado: zoo de cabezas con StandardScaler y hold-out interno`

Ese notebook entrenará todas las cabezas (LogReg L1/L2, RF, ExtraTrees, HistGradBoost, XGBoost, MLP, LightGBM) sobre las features nuevas, con el mismo protocolo (StandardScaler + grid search interno). Y replicará lo mismo sobre las features GAP+GMP originales para que la comparación entre pooling sea limpia.